In [3]:
# ============================================================
# CSET346 NLP LAB 6
# Word Embeddings and Cross-Lingual Text Retrieval
# ============================================================

# INSTALL LIBRARIES
!pip install -q gensim numpy pandas scikit-learn nltk sentence-transformers

import re
import unicodedata
import numpy as np
import pandas as pd
from collections import Counter
from nltk.tokenize import wordpunct_tokenize
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# Q1. WORD2VEC USING CBOW
# ============================================================

english_corpus = [
    "Machine learning helps computers learn from data.",
    "Deep learning uses neural networks.",
    "Artificial intelligence solves complex problems.",
    "Python is a popular programming language.",
    "Students learn programming in college.",
    "Data science combines statistics and computing.",
    "Natural language processing analyzes human language.",
    "Word embeddings represent words as vectors.",
    "Computers process large amounts of data.",
    "Neural networks learn useful patterns.",
    "A computer can perform many calculations.",
    "Programming improves logical thinking.",
    "Python supports machine learning libraries.",
    "Data analysis helps decision making.",
    "Artificial intelligence is used in healthcare.",
    "Robots can perform repetitive tasks.",
    "Algorithms solve computational problems.",
    "Software developers write computer programs.",
    "A database stores organized information.",
    "Cloud computing provides online resources.",
    "Students practice coding every day.",
    "Machine learning models need training data.",
    "Deep neural networks contain many layers.",
    "Natural language processing works with text.",
    "Text classification is an NLP task.",
    "Search engines retrieve relevant information.",
    "Word meaning depends on its context.",
    "Language models predict the next word.",
    "Computer vision analyzes images and videos.",
    "Data preprocessing improves model quality.",
    "Statistics is important for data science.",
    "Artificial intelligence can support education.",
    "A good algorithm is efficient.",
    "Computers understand patterns in data.",
    "Python code is easy to read.",
    "Students develop projects using Python.",
    "Machine learning is used in finance.",
    "Deep learning can recognize objects.",
    "Language translation connects different cultures.",
    "Text documents contain useful information.",
    "Similarity measures compare vector representations.",
    "Cosine similarity compares two vectors.",
    "Word2Vec learns representations from context.",
    "CBOW predicts a word from surrounding words.",
    "GloVe learns from global word co-occurrences.",
    "A vocabulary contains unique words.",
    "Training improves embedding representations.",
    "Sentence vectors represent complete sentences.",
    "Data scientists build predictive models.",
    "Information retrieval finds relevant documents.",
    "Multilingual models support multiple languages."
]

def preprocess_english(text):
    text = text.lower()
    tokens = wordpunct_tokenize(text)
    return [
        token for token in tokens
        if re.fullmatch(r"[a-z]+", token)
    ]

english_tokens = [
    preprocess_english(sentence)
    for sentence in english_corpus
]

# Train CBOW model: sg=0 means CBOW
cbow_model = Word2Vec(
    sentences=english_tokens,
    vector_size=100,
    window=3,
    min_count=1,
    workers=1,
    sg=0,
    epochs=100,
    seed=42
)

print("\n========== Q1: WORD2VEC CBOW ==========")
print("Corpus sentences:", len(english_corpus))
print("Vocabulary size:", len(cbow_model.wv))
print("Vector dimensions:", cbow_model.wv.vector_size)
print("Architecture: CBOW (sg=0)")

selected_words = [
    "machine", "learning", "data", "computer", "python",
    "language", "networks", "model", "student", "programming"
]

print("\n10 Word Vectors:")
for word in selected_words:
    if word in cbow_model.wv:
        print(f"\n{word}: {cbow_model.wv[word]}")
    else:
        print(f"\n{word}: [Not in vocabulary]")

word_pairs = [
    ("machine", "learning"),
    ("computer", "data"),
    ("python", "programming"),
    ("language", "text"),
    ("neural", "networks")
]

print("\nCosine Similarities:")
for w1, w2 in word_pairs:
    if w1 in cbow_model.wv and w2 in cbow_model.wv:
        print(
            f"{w1} - {w2}: "
            f"{cbow_model.wv.similarity(w1, w2):.4f}"
        )
    else:
        print(f"{w1} - {w2}: One of the words is not in vocabulary")

print("\nMost Similar Words:")
for word in ["machine", "learning", "data", "computer", "language"]:
    print(word, ":", cbow_model.wv.most_similar(word, topn=5))


# ============================================================
# Q2. GLOVE EMBEDDINGS AND COMPARISON
# ============================================================

import gensim.downloader as api

print("\n========== Q2: GLOVE ==========")
print("Loading pre-trained GloVe embeddings...")

# Downloads the pre-trained model on first use
glove = api.load("glove-wiki-gigaword-50")

print("GloVe vocabulary size:", len(glove.key_to_index))
print("GloVe vector dimensions:", glove.vector_size)

common_words = [
    "machine", "learning", "data", "computer", "python",
    "language", "networks", "model", "student", "programming"
]

available_words = [
    word for word in common_words
    if word in glove and word in cbow_model.wv
]

print("\nCommon words:", available_words)

print("\nGloVe Word Vectors:")
for word in available_words:
    print(f"\n{word}: {glove[word]}")

print("\nGloVe Cosine Similarities:")
for w1, w2 in word_pairs:
    if w1 in glove and w2 in glove:
        print(
            f"{w1} - {w2}: "
            f"{glove.similarity(w1, w2):.4f}"
        )

print("\nWord2Vec vs GloVe Nearest Neighbours:")
for word in ["machine", "learning", "data", "computer", "language"]:
    print(f"\nWord: {word}")
    print("CBOW:", cbow_model.wv.most_similar(word, topn=5))
    print("GloVe:", glove.most_similar(word, topn=5))

# Optional word analogy
print("\nWord Analogy: king - man + woman")
print(
    glove.most_similar(
        positive=["king", "woman"],
        negative=["man"],
        topn=5
    )
)


# ============================================================
# Q3. HINDI WORD AND SENTENCE REPRESENTATIONS
# ============================================================

hindi_corpus = [
    "मशीन लर्निंग डेटा से सीखने में मदद करती है।",
    "कंप्यूटर कई प्रकार की गणना कर सकता है।",
    "कृत्रिम बुद्धिमत्ता जटिल समस्याओं को हल करती है।",
    "पायथन एक लोकप्रिय प्रोग्रामिंग भाषा है।",
    "विद्यार्थी कॉलेज में प्रोग्रामिंग सीखते हैं।",
    "डेटा विज्ञान में गणित और कंप्यूटिंग शामिल हैं।",
    "प्राकृतिक भाषा प्रसंस्करण मानव भाषा का विश्लेषण करता है।",
    "वर्ड एम्बेडिंग शब्दों को वेक्टर में बदलती है।",
    "कंप्यूटर डेटा को प्रोसेस करते हैं।",
    "न्यूरल नेटवर्क उपयोगी पैटर्न सीखते हैं।",
    "प्रोग्रामिंग तार्किक सोच को बेहतर बनाती है।",
    "पायथन में मशीन लर्निंग लाइब्रेरी उपलब्ध हैं।",
    "डेटा विश्लेषण निर्णय लेने में मदद करता है।",
    "कृत्रिम बुद्धिमत्ता स्वास्थ्य सेवा में उपयोगी है।",
    "रोबोट दोहराए जाने वाले कार्य कर सकते हैं।",
    "एल्गोरिदम समस्याओं को हल करने में मदद करते हैं।",
    "सॉफ्टवेयर डेवलपर कंप्यूटर प्रोग्राम बनाते हैं।",
    "डेटाबेस में जानकारी व्यवस्थित रूप से रखी जाती है।",
    "क्लाउड कंप्यूटिंग ऑनलाइन संसाधन प्रदान करती है।",
    "विद्यार्थी हर दिन कोडिंग का अभ्यास करते हैं।",
    "मशीन लर्निंग मॉडल को ट्रेनिंग डेटा चाहिए।",
    "डीप लर्निंग में कई न्यूरल नेटवर्क लेयर होती हैं।",
    "प्राकृतिक भाषा प्रसंस्करण टेक्स्ट पर काम करता है।",
    "टेक्स्ट वर्गीकरण एनएलपी का एक कार्य है।",
    "सर्च इंजन उपयोगी जानकारी खोजते हैं।",
    "शब्द का अर्थ उसके संदर्भ पर निर्भर करता है।",
    "भाषा मॉडल अगले शब्द का अनुमान लगाते हैं।",
    "कंप्यूटर विजन चित्रों का विश्लेषण करता है।",
    "डेटा प्रीप्रोसेसिंग मॉडल की गुणवत्ता सुधारती है।",
    "सांख्यिकी डेटा विज्ञान के लिए महत्वपूर्ण है।",
    "विद्यार्थी पायथन से प्रोजेक्ट बनाते हैं।",
    "समानता माप दो वेक्टर की तुलना करता है।",
    "वर्ड2वेक शब्दों के संदर्भ से प्रतिनिधित्व सीखता है।",
    "विभिन्न भाषाओं के बीच अनुवाद उपयोगी है।",
    "वेक्टर पूरे वाक्य का प्रतिनिधित्व कर सकते हैं।"
]

def preprocess_hindi(text):
    text = unicodedata.normalize("NFC", text)
    tokens = wordpunct_tokenize(text)
    return [
        token for token in tokens
        if re.search(r"[\u0900-\u097F]", token)
    ]

hindi_tokens = [
    preprocess_hindi(sentence)
    for sentence in hindi_corpus
]

print("\n========== Q3: HINDI EMBEDDINGS ==========")
print("Hindi sentences:", len(hindi_corpus))
print("Original:", hindi_corpus[0])
print("Processed:", hindi_tokens[0])

hindi_model = Word2Vec(
    sentences=hindi_tokens,
    vector_size=100,
    window=3,
    min_count=1,
    workers=1,
    sg=0,
    epochs=100,
    seed=42
)

print("Hindi vocabulary size:", len(hindi_model.wv))
print("Hindi vector dimensions:", hindi_model.wv.vector_size)

word_counts = Counter(
    word for sentence in hindi_tokens for word in sentence
)

frequent_words = [
    word for word, count in word_counts.most_common(10)
    if word in hindi_model.wv
]

print("\n10 Frequent Hindi Word Vectors:")
for word in frequent_words:
    print(f"\n{word}: {hindi_model.wv[word]}")

hindi_pairs = [
    ("मशीन", "लर्निंग"),
    ("कंप्यूटर", "डेटा"),
    ("प्रोग्रामिंग", "भाषा"),
    ("प्राकृतिक", "भाषा"),
    ("न्यूरल", "नेटवर्क")
]

print("\nHindi Word Similarity:")
for w1, w2 in hindi_pairs:
    if w1 in hindi_model.wv and w2 in hindi_model.wv:
        print(
            f"{w1} - {w2}: "
            f"{hindi_model.wv.similarity(w1, w2):.4f}"
        )
    else:
        print(f"{w1} - {w2}: word not found")

# Sentence vector = average of word vectors
def hindi_sentence_vector(sentence):
    tokens = preprocess_hindi(sentence)
    vectors = [
        hindi_model.wv[token]
        for token in tokens
        if token in hindi_model.wv
    ]

    if not vectors:
        return np.zeros(hindi_model.wv.vector_size)

    return np.mean(vectors, axis=0)

sentence_pairs = [
    (0, 10),
    (1, 8),
    (2, 13),
    (3, 11),
    (6, 22)
]

print("\nHindi Sentence Similarity:")
for i, j in sentence_pairs:
    v1 = hindi_sentence_vector(hindi_corpus[i])
    v2 = hindi_sentence_vector(hindi_corpus[j])

    score = cosine_similarity(
        v1.reshape(1, -1),
        v2.reshape(1, -1)
    )[0][0]

    print(f"\nSentence 1: {hindi_corpus[i]}")
    print(f"Sentence 2: {hindi_corpus[j]}")
    print(f"Cosine similarity: {score:.4f}")

print("Word vector dimensions:", hindi_model.wv.vector_size)
print(
    "Sentence vector dimensions:",
    hindi_sentence_vector(hindi_corpus[0]).shape[0]
)


# ============================================================
# Q4. PARALLEL CORPUS AND CROSS-LINGUAL RETRIEVAL
# ============================================================

from sentence_transformers import SentenceTransformer

english_sentences = [
    "Machine learning learns from data.",
    "A computer processes information.",
    "Python is a programming language.",
    "Students study at college.",
    "Artificial intelligence solves problems.",
    "The sun rises in the east.",
    "Water is essential for life.",
    "I like reading books.",
    "The weather is very pleasant.",
    "We should protect the environment.",
    "Education improves our knowledge.",
    "The train arrived at the station.",
    "She is drinking a glass of water.",
    "Farmers grow crops in the fields.",
    "The child is playing in the park.",
    "The teacher explains the lesson.",
    "India is a diverse country.",
    "Technology changes our lives.",
    "The library contains many books.",
    "Exercise keeps the body healthy."
]

hindi_sentences = [
    "मशीन लर्निंग डेटा से सीखती है।",
    "कंप्यूटर जानकारी को प्रोसेस करता है।",
    "पायथन एक प्रोग्रामिंग भाषा है।",
    "विद्यार्थी कॉलेज में पढ़ते हैं।",
    "कृत्रिम बुद्धिमत्ता समस्याओं को हल करती है।",
    "सूरज पूर्व दिशा में उगता है।",
    "पानी जीवन के लिए आवश्यक है।",
    "मुझे किताबें पढ़ना पसंद है।",
    "मौसम बहुत सुहावना है।",
    "हमें पर्यावरण की रक्षा करनी चाहिए।",
    "शिक्षा हमारे ज्ञान को बढ़ाती है।",
    "ट्रेन स्टेशन पर पहुँची।",
    "वह एक गिलास पानी पी रही है।",
    "किसान खेतों में फसल उगाते हैं।",
    "बच्चा पार्क में खेल रहा है।",
    "शिक्षक पाठ समझाते हैं।",
    "भारत एक विविधताओं वाला देश है।",
    "तकनीक हमारे जीवन को बदलती है।",
    "पुस्तकालय में बहुत सी किताबें हैं।",
    "व्यायाम शरीर को स्वस्थ रखता है।"
]

print("\n========== Q4: CROSS-LINGUAL RETRIEVAL ==========")
assert len(english_sentences) == len(hindi_sentences)
print("Aligned sentence pairs:", len(english_sentences))

# Shared multilingual embedding space
print("Loading multilingual Sentence Transformer...")
multilingual_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

english_vectors = multilingual_model.encode(
    english_sentences,
    convert_to_numpy=True,
    normalize_embeddings=True
)

hindi_vectors = multilingual_model.encode(
    hindi_sentences,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("English vector shape:", english_vectors.shape)
print("Hindi vector shape:", hindi_vectors.shape)

# Compare every English sentence with every Hindi sentence
similarity_matrix = cosine_similarity(
    english_vectors,
    hindi_vectors
)

print("Similarity matrix shape:", similarity_matrix.shape)

query_indices = [0, 4, 7, 12, 19]
recall_at_1 = 0
recall_at_5 = 0

for query_idx in query_indices:
    scores = similarity_matrix[query_idx]
    ranked_indices = np.argsort(scores)[::-1]

    # The aligned translation is at the same index
    correct_idx = query_idx
    correct_rank = int(
        np.where(ranked_indices == correct_idx)[0][0]
    ) + 1

    if correct_rank == 1:
        recall_at_1 += 1

    if correct_rank <= 5:
        recall_at_5 += 1

    print("\n" + "=" * 60)
    print("English query:", english_sentences[query_idx])
    print("Correct Hindi:", hindi_sentences[correct_idx])
    print("Correct translation rank:", correct_rank)
    print(
        "Correct translation similarity:",
        round(float(scores[correct_idx]), 4)
    )

    print("\nTop 5 retrieved Hindi sentences:")
    for rank, idx in enumerate(ranked_indices[:5], start=1):
        print(
            f"Rank {rank}: {hindi_sentences[idx]} "
            f"| Similarity: {scores[idx]:.4f}"
        )

total_queries = len(query_indices)

print("\nRecall@1:", recall_at_1 / total_queries)
print("Recall@5:", recall_at_5 / total_queries)

# Full retrieval table for all 20 queries
results = []

for i, query in enumerate(english_sentences):
    ranked_indices = np.argsort(similarity_matrix[i])[::-1]
    top_idx = ranked_indices[0]

    correct_rank = int(
        np.where(ranked_indices == i)[0][0]
    ) + 1

    results.append({
        "English Query": query,
        "Top Retrieved Hindi": hindi_sentences[top_idx],
        "Top Similarity": round(
            float(similarity_matrix[i, top_idx]), 4
        ),
        "Correct Translation Rank": correct_rank,
        "Correct at Rank 1": correct_rank == 1
    })

results_df = pd.DataFrame(results)

print("\nFull Retrieval Results:")
print(results_df.to_string(index=False))

results_df.to_csv(
    "cross_lingual_retrieval_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nResults saved to cross_lingual_retrieval_results.csv")
print("\n========== ALL FOUR QUESTIONS COMPLETED ==========")


========== Q1: WORD2VEC CBOW ==========
Corpus sentences: 51
Vocabulary size: 167
Vector dimensions: 100
Architecture: CBOW (sg=0)

10 Word Vectors:

machine: [ 0.02351258 -0.00010155  0.00110004  0.00416616  0.00067     0.0153402
  0.00902033  0.0108417  -0.0197914   0.01451643 -0.01371902  0.0162136
  0.00468884  0.02570753 -0.00604738 -0.01760005  0.00278492 -0.01536407
 -0.01268598 -0.00563869  0.01241163  0.00893773  0.01844284  0.00672214
  0.01359466  0.00592497 -0.01362705  0.01751249 -0.00833717 -0.01434853
  0.00117748  0.00433878  0.01387642  0.00537027  0.00122587 -0.0070655
 -0.00266619 -0.02487603 -0.00028192  0.00010921  0.00581784  0.00277716
  0.02620234 -0.0152209   0.01286338  0.00376261  0.01394712  0.00321437
  0.01571498  0.0033831  -0.01353761 -0.02950908 -0.00298707  0.0054218
  0.01713513 -0.01110184 -0.00662217  0.01620081 -0.003924    0.01452236
 -0.00490469  0.01445143 -0.01318501  0.01698472  0.00178752 -0.00389306
 -0.00671603 -0.01869696  0.00516298 -0.0

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

English vector shape: (20, 384)
Hindi vector shape: (20, 384)
Similarity matrix shape: (20, 20)

English query: Machine learning learns from data.
Correct Hindi: मशीन लर्निंग डेटा से सीखती है।
Correct translation rank: 1
Correct translation similarity: 0.894

Top 5 retrieved Hindi sentences:
Rank 1: मशीन लर्निंग डेटा से सीखती है। | Similarity: 0.8940
Rank 2: शिक्षा हमारे ज्ञान को बढ़ाती है। | Similarity: 0.4697
Rank 3: कंप्यूटर जानकारी को प्रोसेस करता है। | Similarity: 0.4613
Rank 4: कृत्रिम बुद्धिमत्ता समस्याओं को हल करती है। | Similarity: 0.3951
Rank 5: शिक्षक पाठ समझाते हैं। | Similarity: 0.3778

English query: Artificial intelligence solves problems.
Correct Hindi: कृत्रिम बुद्धिमत्ता समस्याओं को हल करती है।
Correct translation rank: 1
Correct translation similarity: 0.9617

Top 5 retrieved Hindi sentences:
Rank 1: कृत्रिम बुद्धिमत्ता समस्याओं को हल करती है। | Similarity: 0.9617
Rank 2: मशीन लर्निंग डेटा से सीखती है। | Similarity: 0.4639
Rank 3: कंप्यूटर जानकारी को प्रोसेस करता है।